# RECCURENT NEURAL NETWORK (RNN) FOR SENTIMENT ANALYSIS

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [36]:
data = pd.read_csv("IMDB Dataset.csv")
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [37]:
data.isnull().sum()

review       0
sentiment    0
dtype: int64

In [38]:
data.shape

(50000, 2)

In [39]:
data.drop_duplicates(inplace=True)
data.shape

(49582, 2)

In [40]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


# Text Pre_processing

In [42]:
#1. Converting into the lower case

data["review"] = data["review"].str.lower()


#2. Removing the URLs

import re # Regular Expression Operations library

def remove_urls(text):
    text_new = re.sub(r"http\S+", "", text) # (pattern, replacement, string)
    return text_new

data["review"] = data["review"].apply(remove_urls)

#3. Removing Punctuations
def remove_punctuations(text):
    text_new = re.sub(r"[^A-Za-z0-9\s]", "", text)  # ^ means excluding the next content# here we keep A-Z, a-z and 0-9 and spaces and remove any things in the text which are basicall the punctuations
    return text_new

data["review"] = data["review"].apply(remove_punctuations)

#4. Removing the HTML tags

def remove_HTMLs(text):
    text_new = re.sub(r"<.*?>", "", text) # <.*?> this means . all the characters, * no matter how many, ? in a gready way thats means that from < to > : remove all the in this bracket

    return text_new

data["review"] = data["review"].apply(remove_HTMLs)

#5. Removing the stop words

import nltk # a very important library for natural language toolkit used for performing different operations like tokenization and stopwords etc. 

nltk.download("punkt") # punkt is the official tokenizer in the nltk
nltk.download("punkt_tab")
nltk.download("stopwords")

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopwords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")

    for words in tokens:
        if words in stop_words:
            text = text.replace(words, "")
            return text
data["review"] = data["review"].apply(remove_stopwords)


[nltk_data] Downloading package punkt to /Users/muhammad/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/muhammad/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/muhammad/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [50]:
### 6. Stemming
# played >> play
# Coding >> code
# porterstemming
data = data.dropna(subset=["review"]) # code is crashing due to one missing value after html removal, that what add this line
from nltk.stem import PorterStemmer

def stemming(text):
    stemmed_words = []
    ps = PorterStemmer()
    tokens = word_tokenize(text)
    for token in tokens:
        stemmed = ps.stem(token)
        stemmed_words.append(stemmed)
    return " ".join(stemmed_words)  # to convert them into the string

data["review"] = data["review"].apply(stemming)

/var/folders/03/rq1lfm0x7pjfw_g6wj1hv1wc0000gn/T/ipykernel_3331/2153613646.py:17: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data["review"] = data["review"].apply(stemming)


In [52]:
data.head()

,review,sentiment
0,one the other review ha mention that after wat...,positive
1,wonder littl product br br the film techniqu i...,positive
2,thought th wa a wonder way to spend tme on a t...,positive
3,bsiclli there fmili where littl boy jke think ...,negative
4,petter mattei love the time of money is a visu...,positive


In [59]:
#7. To encode the ouput
from sklearn.preprocessing import LabelEncoder
lb = LabelEncoder()
y = lb.fit_transform(data["sentiment"])

In [55]:
# 8. Vectarization
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)
x = tf.fit_transform(data["review"])



# Datasets and Dataloaders

In [80]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x, y, random_state=42, test_size=0.2)

import torch
from torch.utils.data import DataLoader, TensorDataset
# here our daata is in sparse matrix so we need to convert it into the numpy array for datasets
x_train = x_train.toarray()
x_test = x_test.toarray()

train_set = TensorDataset(
    torch.from_numpy(x_train).float(),
    torch.tensor(y_train).float()
)

test_set = TensorDataset(
    torch.from_numpy(x_test).float(),
    torch.tensor(y_test).float()
)

train_loader = DataLoader(train_set, shuffle=True, batch_size=32)
test_loader = DataLoader(test_set)


# BUILD RNN


In [83]:
import torch
import torch.nn as nn
import torch.optim as optim


class RNN (nn.Module):
    def __init__(self, input_size, hidden_size = 128, num_layers = 1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
# RNN layer
        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True) # pytorch expect(sequence_leng, batch_size, inp_size) batch_first = Ture will help us to send batch_size first and then the sequence length, this will help us to reduce the number of reshaing tasks that we have to comuute multiple times

# fully connected layer
        self.fc = nn.Linear(hidden_size, 1)
        
    def forward(self, x):
    # optinoal: hidden layer initialization. >> shape(num of layers, batch_size, hidden_size)
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out,_ = self.rnn(x, h0)
        # 1st value = hidden state of all the timestemps >>(batch, sequence_len, hidden_size)
        # 2st value = hidden state of the last timestemps
        out = self.fc(out[:,-1,:])
        return out

In [86]:
input_size = x_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()  # binary croos entropy loss
optimizer = optim.Adam(model.parameters())


## Train RNN Model

In [93]:
epocs = 10 
for epoc in range(epocs):
    model.train()
# with sequeeze or unsequeeze functions we basically reduce or increase the dimmention of data
    for xb, yb in train_loader:
        optimizer.zero_grad()
        xb = xb.unsqueeze(1) # add singleton direction because the model is expecting a three dimmentional data
        outputs = model(xb) # (batch size, 1)
        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,1)  but the sigmoid expect a one dimmentional ==> probablility
        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backpropagation
        optimizer.step() # update weights
        
    print(f"epoc = {epoc} / {epocs} and loss = {loss.item()}") 

epoc = 0 / 10 and loss = 0.03569476306438446
epoc = 1 / 10 and loss = 0.06560777127742767
epoc = 2 / 10 and loss = 0.42131921648979187
epoc = 3 / 10 and loss = 0.38198596239089966
epoc = 4 / 10 and loss = 0.3409324586391449
epoc = 5 / 10 and loss = 0.056950777769088745
epoc = 6 / 10 and loss = 0.24464666843414307
epoc = 7 / 10 and loss = 0.1122802197933197
epoc = 8 / 10 and loss = 0.4723082184791565
epoc = 9 / 10 and loss = 0.21977749466896057


In [96]:
# model evaluation
model.eval()
with torch.no_grad():
    correct = 0
    tot_val = 0

    for xb, yb in test_loader:
        xb = xb.unsqueeze(1)
        outputs = model(xb)
        predicted = (torch.sigmoid(outputs.squeeze())>0.5).float()

        tot_val += yb.size(0)
        correct += (predicted== yb).sum().item()
    print(f"accuracy = {correct/tot_val *100} %")
          
        


accuracy = 86.2155893919532 %
